### Import dependencies

In [2]:
import openai
import os
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langsmith import Client
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


In [3]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

### Download example eval dataset from Langsmith



In [4]:
ls_client = Client()

In [ ]:
dataset = ls_client.read_dataset(dataset_name="rag-eval-dataset")

In [14]:
list(ls_client.list_examples(dataset=dataset.id, limit=50))[12].outputs

{'ground_truth': 'Yes, there is a portable AM/FM radio that runs on 2 AA batteries and includes a 3.5mm headphone jack. It also has a tuning light, back clip, adjustable antenna, and is designed for both indoor and outdoor use.',
 'retrieved_context_ids': ['B0C8S6BBY9'],
 'reference_descriptions': ["PRUNUS J-166 Portable Radio AM FM, Battery Operated with Tuning Light, Back Clip, Excellent Reception for Indoor & Outdoor Emergency Radio, FM Portable, Transistor 【Portable Design】You can hold mini radio by one hand. it's small enough to put it in emergency kit. This FM AM radio measures 69*128*28mm with weight only 119g (without batteries), close to the size of iPhone 7. More importantly, thanks to its back clip and lanyard, transistor radio is pretty easy to carry around whether it’s clipped to pocket or carried with lanyard. Now, take it for morning exercise, stroll or a break in park, together with a piece of brisk music or a great radio show. 【Easy to Use】 After busy work, sometimes i

In [16]:
list(ls_client.list_examples(dataset=dataset.id, limit=50))[12].inputs

{'question': 'Do you sell a small battery-operated AM/FM radio with a headphone jack?'}

In [17]:
reference_input = list(ls_client.list_examples(dataset=dataset.id, limit=50))[12].inputs
reference_output = list(ls_client.list_examples(dataset=dataset.id, limit=50))[12].outputs

### RAG Pipeline

In [18]:
qdrant_client = QdrantClient(url='http://localhost:6333')

def get_embedding(text, model='text-embedding-3-small'):
    response = openai.embeddings.create(
        model=model,
        input=text,
    )
    
    return response.data[0].embedding

def retrieve_data(query, collection_name='amazon-items-collection-01', k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context_scores = []
    retrieved_context_texts = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context_scores.append(result.score)
        retrieved_context_texts.append(result.payload['processed_description'])
        retrieved_context_ratings.append(result.payload['average_rating'])

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieved_context_scores': retrieved_context_scores,
        'retrieved_context_texts': retrieved_context_texts,
        'retrieved_context_ratings': retrieved_context_ratings
    }

def process_context(retrieve_context):
    formatted_context = ''

    for id, chunk, rating in zip(retrieve_context['retrieved_context_ids'], retrieve_context['retrieved_context_texts'], retrieve_context['retrieved_context_ratings']):
        formatted_context += f"- Product ID: {id}, Product Rating: {rating}, Product Description: {chunk}\n"

    return formatted_context

def build_prompt(question, formatted_context):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting

    Context:
    {formatted_context}

    Question:
    {question}
    """
    return prompt

def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort='none'
    )

    return response.choices[0].message.content

def rag_pipeline(question, topk=5):
    retrieved_context = retrieve_data(query=question, k=topk)
    formatted_context = process_context(retrieved_context)
    prompt = build_prompt(question, formatted_context)
    answer = generate_answer(prompt)
    
    final_answer = {
        'answer': answer,
        'question': question,
        'retrieved_context_ids': retrieved_context['retrieved_context_ids'],
        'retrieved_context': retrieved_context['retrieved_context_texts'],
    }

    return final_answer

### RAGAS metrics (https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/)

In [22]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

/tmp/ipykernel_24957/3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/tmp/ipykernel_24957/3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/tmp/ipykernel_24957/3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Fa

In [23]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

/tmp/ipykernel_24957/840510326.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
/tmp/ipykernel_24957/840510326.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [26]:
result = rag_pipeline(reference_input['question'])

In [27]:
result

{'answer': 'Yes. PRUNUS J-166 Portable Radio (Product ID: B0C8S6BBY9) is a small battery-operated AM/FM radio (2x AA batteries, not included) and it has a 3.5mm earphone jack for headphones.',
 'question': 'Do you sell a small battery-operated AM/FM radio with a headphone jack?',
 'retrieved_context_ids': ['B0C8S6BBY9',
  'B0CH8DRD6K',
  'B09KQP2H7N',
  'B0BBF2VC6X',
  'B0C4NJPN4Q'],
 'retrieved_context': ["PRUNUS J-166 Portable Radio AM FM, Battery Operated with Tuning Light, Back Clip, Excellent Reception for Indoor & Outdoor Emergency Radio, FM Portable, Transistor 【Portable Design】You can hold mini radio by one hand. it's small enough to put it in emergency kit. This FM AM radio measures 69*128*28mm with weight only 119g (without batteries), close to the size of iPhone 7. More importantly, thanks to its back clip and lanyard, transistor radio is pretty easy to carry around whether it’s clipped to pocket or carried with lanyard. Now, take it for morning exercise, stroll or a break i

In [28]:
reference_output

{'ground_truth': 'Yes, there is a portable AM/FM radio that runs on 2 AA batteries and includes a 3.5mm headphone jack. It also has a tuning light, back clip, adjustable antenna, and is designed for both indoor and outdoor use.',
 'retrieved_context_ids': ['B0C8S6BBY9'],
 'reference_descriptions': ["PRUNUS J-166 Portable Radio AM FM, Battery Operated with Tuning Light, Back Clip, Excellent Reception for Indoor & Outdoor Emergency Radio, FM Portable, Transistor 【Portable Design】You can hold mini radio by one hand. it's small enough to put it in emergency kit. This FM AM radio measures 69*128*28mm with weight only 119g (without batteries), close to the size of iPhone 7. More importantly, thanks to its back clip and lanyard, transistor radio is pretty easy to carry around whether it’s clipped to pocket or carried with lanyard. Now, take it for morning exercise, stroll or a break in park, together with a piece of brisk music or a great radio show. 【Easy to Use】 After busy work, sometimes i

In [34]:
async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run['retrieved_context_ids'],
        reference_context_ids=example['retrieved_context_ids'],
    )
    
    scorer = IDBasedContextPrecision()
    return await scorer.single_turn_ascore(sample)

In [35]:
await ragas_context_precision_id_based(result, reference_output) #1 out of 5

0.2

In [36]:
async def ragas_context_recall_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run['retrieved_context_ids'],
        reference_context_ids=example['retrieved_context_ids'],
    )
    
    scorer = IDBasedContextRecall()
    return await scorer.single_turn_ascore(sample)

In [38]:
await ragas_context_recall_id_based(result, reference_output) #the chunk ID returned from the pipeline is in the reference context ids

1.0

In [46]:
async def ragas_faithfullness(run):
    sample = SingleTurnSample(
        user_input=run['question'],
        response=run['answer'],
        retrieved_contexts=run['retrieved_context'],
    )
    
    scorer = Faithfulness(llm=ragas_llm)
    return await scorer.single_turn_ascore(sample)

In [47]:
await ragas_faithfullness(result) #indicates that the answer is faithful to the context (can be supported by the retrieved context)

1.0

In [48]:
async def ragas_answer_relevancy(run):
    sample = SingleTurnSample(
        user_input=run['question'],
        response=run['answer'],
        retrieved_contexts=run['retrieved_context'],
    )
    
    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
    return await scorer.single_turn_ascore(sample)

In [49]:
await ragas_answer_relevancy(result) #indicates the level of relevance of the answer to the question

np.float64(0.45358723456648914)